In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Locate the project without a personal username, drive, or fixed notebook depth.
REPO_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data/raw/home-credit-default-risk").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the project and extract the Kaggle CSVs "
        "into data/raw/home-credit-default-risk/."
    )
DATA_RAW = REPO_ROOT / "data/raw/home-credit-default-risk"
DATA_INTERIM = REPO_ROOT / "data/interim"
DATA_PROCESSED = REPO_ROOT / "data/processed"
RAW_DIR = DATA_RAW
INTERIM_DIR = DATA_INTERIM
PROCESSED_DIR = DATA_PROCESSED


In [3]:
feature_files = {
    "bureau": "bureau_features.parquet",
    "previous_application": "previous_application_features.parquet",
    "pos_cash": "pos_cash_features.parquet",
    "installments": "installments_features.parquet",
    "credit_card": "credit_card_features.parquet",
}

In [4]:
print("Interim directory:", INTERIM_DIR.resolve())

Interim directory: E:\Depi Notes Data Engineering\DEPI project R5\ProJect\data\interim


In [5]:
features = {}

for name, filename in feature_files.items():
    path = INTERIM_DIR / filename
    features[name] = pd.read_parquet(path)

    print(f"{name}: {features[name].shape}")


bureau: (305811, 32)
previous_application: (338857, 56)
pos_cash: (338857, 89)
installments: (338857, 97)
credit_card: (338857, 136)


In [6]:
for name, df in features.items():
    print(f"\n--- {name} ---")
    print("Rows:", len(df))
    print("Unique applicants:", df["SK_ID_CURR"].nunique())
    print("Duplicate IDs:", df["SK_ID_CURR"].duplicated().sum())
    print("Missing IDs:", df["SK_ID_CURR"].isna().sum())
    print("Columns:", len(df.columns))


--- bureau ---
Rows: 305811
Unique applicants: 305811
Duplicate IDs: 0
Missing IDs: 0
Columns: 32

--- previous_application ---
Rows: 338857
Unique applicants: 338857
Duplicate IDs: 0
Missing IDs: 0
Columns: 56

--- pos_cash ---
Rows: 338857
Unique applicants: 338857
Duplicate IDs: 0
Missing IDs: 0
Columns: 89

--- installments ---
Rows: 338857
Unique applicants: 338857
Duplicate IDs: 0
Missing IDs: 0
Columns: 97

--- credit_card ---
Rows: 338857
Unique applicants: 338857
Duplicate IDs: 0
Missing IDs: 0
Columns: 136


In [7]:
feature_columns = {
    name: set(df.columns) - {"SK_ID_CURR"}
    for name, df in features.items()
}

all_feature_names = []

for name, columns in feature_columns.items():
    all_feature_names.extend(columns)

duplicates = pd.Series(all_feature_names)
duplicates = duplicates[duplicates.duplicated()].unique()

print("Total historical features:", len(all_feature_names))
print("Unique feature names:", len(set(all_feature_names)))
print("Duplicate feature names:", list(duplicates))


Total historical features: 405
Unique feature names: 405
Duplicate feature names: []


In [8]:
id_sets = {
    name: set(df["SK_ID_CURR"])
    for name, df in features.items()
}

# Compare the four tables with 338,857 applicants
reference_ids = id_sets["previous_application"]

for name, ids in id_sets.items():
    print(f"\n{name}")
    print("Applicants:", len(ids))
    print("In reference but missing here:", len(reference_ids - ids))
    print("Additional applicants:", len(ids - reference_ids))

# Bureau coverage
print("\nBureau coverage:")
print("Bureau applicants:", len(id_sets["bureau"]))
print("Applicants in reference:", len(reference_ids))
print("Bureau applicants outside reference:",
      len(id_sets["bureau"] - reference_ids))
print("Reference applicants without Bureau:",
      len(reference_ids - id_sets["bureau"]))



bureau
Applicants: 305811
In reference but missing here: 47766
Additional applicants: 14720

previous_application
Applicants: 338857
In reference but missing here: 0
Additional applicants: 0

pos_cash
Applicants: 338857
In reference but missing here: 0
Additional applicants: 0

installments
Applicants: 338857
In reference but missing here: 0
Additional applicants: 0

credit_card
Applicants: 338857
In reference but missing here: 0
Additional applicants: 0

Bureau coverage:
Bureau applicants: 305811
Applicants in reference: 338857
Bureau applicants outside reference: 14720
Reference applicants without Bureau: 47766


In [9]:
application_train = pd.read_csv(
    RAW_DIR / "application_train.csv"
)

application_test = pd.read_csv(
    RAW_DIR / "application_test.csv"
)

print("Train shape:", application_train.shape)
print("Test shape:", application_test.shape)

print("Train unique IDs:", application_train["SK_ID_CURR"].nunique())
print("Test unique IDs:", application_test["SK_ID_CURR"].nunique())

print("Train duplicate IDs:", application_train["SK_ID_CURR"].duplicated().sum())
print("Test duplicate IDs:", application_test["SK_ID_CURR"].duplicated().sum())

Train shape: (307511, 122)
Test shape: (48744, 121)
Train unique IDs: 307511
Test unique IDs: 48744
Train duplicate IDs: 0
Test duplicate IDs: 0


In [10]:
competition_ids = set(application_train["SK_ID_CURR"]) | set(
    application_test["SK_ID_CURR"]
)

print("Total competition applicants:", len(competition_ids))

for name, ids in id_sets.items():
    matched = len(competition_ids & ids)

    print(f"\n{name}")
    print("Historical applicants:", len(ids))
    print("Matched to competition:", matched)
    print("Not in competition:", len(ids - competition_ids))
    print("Competition applicants without history:", len(competition_ids - ids))

Total competition applicants: 356255

bureau
Historical applicants: 305811
Matched to competition: 305811
Not in competition: 0
Competition applicants without history: 50444

previous_application
Historical applicants: 338857
Matched to competition: 338857
Not in competition: 0
Competition applicants without history: 17398

pos_cash
Historical applicants: 338857
Matched to competition: 338857
Not in competition: 0
Competition applicants without history: 17398

installments
Historical applicants: 338857
Matched to competition: 338857
Not in competition: 0
Competition applicants without history: 17398

credit_card
Historical applicants: 338857
Matched to competition: 338857
Not in competition: 0
Competition applicants without history: 17398


In [11]:
train_ids = set(application_train["SK_ID_CURR"])
test_ids = set(application_test["SK_ID_CURR"])

print("Train applicants:", len(train_ids))
print("Test applicants:", len(test_ids))
print("Train/test overlap:", len(train_ids & test_ids))
print("Combined applicants:", len(train_ids | test_ids))

Train applicants: 307511
Test applicants: 48744
Train/test overlap: 0
Combined applicants: 356255


In [12]:
all_columns = []

for name, df in features.items():
    columns = [col for col in df.columns if col != "SK_ID_CURR"]
    all_columns.extend(columns)

duplicate_columns = pd.Series(all_columns)
duplicate_columns = duplicate_columns[
    duplicate_columns.duplicated(keep=False)
]

print("Total historical features:", len(all_columns))
print("Unique feature names:", len(set(all_columns)))
print("Duplicate feature names:", sorted(duplicate_columns.unique()))

Total historical features: 405
Unique feature names: 405
Duplicate feature names: []


In [13]:
# Use competition IDs only as the applicant population.
# We do not merge application features or TARGET.

competition_ids = pd.concat([
    application_train[["SK_ID_CURR"]],
    application_test[["SK_ID_CURR"]]
], ignore_index=True)

competition_ids = competition_ids.drop_duplicates(
    subset="SK_ID_CURR"
)

print("Competition applicants:", len(competition_ids))
print("Unique IDs:", competition_ids["SK_ID_CURR"].nunique())

Competition applicants: 356255
Unique IDs: 356255


In [14]:
historical_features = competition_ids.copy()

for name, df in features.items():
    historical_features = historical_features.merge(
        df,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    print(f"Merged: {name}")

print("\nFinal shape:", historical_features.shape)
print("Unique IDs:", historical_features["SK_ID_CURR"].nunique())
print("Duplicate IDs:", historical_features["SK_ID_CURR"].duplicated().sum())

Merged: bureau
Merged: previous_application
Merged: pos_cash
Merged: installments
Merged: credit_card

Final shape: (356255, 406)
Unique IDs: 356255
Duplicate IDs: 0


In [15]:
assert set(historical_features["SK_ID_CURR"]) == set(
    competition_ids["SK_ID_CURR"]
)

assert historical_features.drop(columns="SK_ID_CURR").shape[1] == 405

print("All final validation checks passed.")

All final validation checks passed.


In [16]:
output_path = PROCESSED_DIR / "historical_features.parquet"

historical_features.to_parquet(output_path, index=False)

print("Saved:", output_path.resolve())
print("Shape:", historical_features.shape)

Saved: E:\Depi Notes Data Engineering\DEPI project R5\ProJect\data\processed\historical_features.parquet
Shape: (356255, 406)


In [17]:
hf_location = PROCESSED_DIR / "historical_features.parquet"

hf = pd.read_parquet(hf_location)
hf.shape

(356255, 406)

In [18]:
assert historical_features.shape == (356255, 406)
assert historical_features["SK_ID_CURR"].is_unique
assert historical_features["SK_ID_CURR"].notna().all()
assert historical_features.columns.is_unique

print("Historical features dataset validated successfully!")

Historical features dataset validated successfully!
